# One Function

In [3]:
import os
import random
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score
import joblib
import nltk
nltk.download(['punkt', 'wordnet'])
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

# Function to tokenize text
def tokenize(text_data):
    tokens = word_tokenize(text_data)
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(tok).lower().strip() for tok in tokens]

# Function to vectorize and transform data
def vectorize_transform(df):
    count_vectorizer = CountVectorizer(tokenizer=tokenize, token_pattern=None, ngram_range=(1, 3))
    tfidf_transformer = TfidfTransformer()
    text_data_counts = count_vectorizer.fit_transform(df)
    text_data_tfidf = tfidf_transformer.fit_transform(text_data_counts)
    return text_data_tfidf, count_vectorizer, tfidf_transformer

# Function to identify tail labels
def get_tail_label(df):
    columns = df.columns
    n = len(columns)
    irpl = np.zeros(n)
    for column in range(n):
        irpl[column] = df[columns[column]].value_counts()[1]
    irpl = max(irpl) / irpl
    mir = np.average(irpl)
    tail_label = [columns[i] for i in range(n) if irpl[i] > mir]
    return tail_label

# Function to get indices of tail labels
def get_index(df):
    tail_labels = get_tail_label(df)
    index = set()
    for tail_label in tail_labels:
        sub_index = set(df[df[tail_label] == 1].index)
        index = index.union(sub_index)
    return list(index)

# Function to get minority instances
def get_minority_instance(X, y):
    index = get_index(y)
    X_sub = X[X.index.isin(index)].reset_index(drop=True)
    y_sub = y[y.index.isin(index)].reset_index(drop=True)
    return X_sub, y_sub

# Function to find nearest neighbors
def nearest_neighbour(X):
    nbs = NearestNeighbors(n_neighbors=5, metric='euclidean', algorithm='kd_tree').fit(X)
    _, indices = nbs.kneighbors(X)
    return indices

# Function to apply MLSMOTE algorithm
def MLSMOTE(X, y, n_sample, indices):
    X = pd.DataFrame(X.toarray())
    n = len(indices)
    new_X = np.zeros((n_sample, X.shape[1]))
    target = np.zeros((n_sample, y.shape[1]))

    for i in range(n_sample):
        reference = random.randint(0, n - 1)
        neighbour = random.choice(indices[reference, 1:])
        all_point = indices[reference]
        nn_df = y[y.index.isin(all_point)]
        ser = nn_df.sum(axis=0, skipna=True)
        target[i] = np.array([1 if val > 2 else 0 for val in ser])
        ratio = random.random()
        gap = X.loc[reference, :] - X.loc[neighbour, :]
        new_X[i] = np.array(X.loc[reference, :] + ratio * gap)

    new_X = pd.DataFrame(new_X, columns=X.columns)
    target = pd.DataFrame(target, columns=y.columns)
    target = pd.concat([y, target], axis=0)
    new_X = pd.concat([X, new_X], axis=0)
    new_X = csr_matrix(new_X.values)
    return new_X, target

# Function to vectorize test data
def vectorize_test(text_data, count_vectorizer, tfidf_transformer):
    text_data_counts = count_vectorizer.transform(text_data)
    text_data_tfidf = tfidf_transformer.transform(text_data_counts)
    return text_data_tfidf

def main():
    # Move to datasets folder
    original_directory = os.getcwd()
    dataset_directory = './dataset'
    os.chdir(dataset_directory)

    # Import data
    engine = create_engine('sqlite:///DisasterResponse.db')
    with engine.connect() as connection:
        df = pd.read_sql("SELECT * FROM messages", connection)
    engine.dispose()

    # Return to original directory
    os.chdir(original_directory)

    # Get target columns
    target_columns = [col for col in df.columns if col not in ['message', 'related', 'id', 'original', 'genre']]
    df_related = df[df['related'] == 1]
    columns_to_drop = df_related[target_columns].sum() == 0
    columns_to_drop = columns_to_drop[columns_to_drop].index

    # New datasets for multi-label classification
    X = df_related['message']
    Y = df_related[target_columns]
    Y = Y.drop(columns=columns_to_drop, axis=1)

    # Split the dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

    # Get minority instance (tail labels) of that dataframe
    X_sub, y_sub = get_minority_instance(X_train, y_train)

    # Vectorize the features
    X_tfidf, count_vectorizer, tfidf_transformer = vectorize_transform(X_sub)

    # Get index of 5 nearest neighbors of all the instances
    indices = nearest_neighbour(X_tfidf)

    # Apply MLSMOTE to augment the dataframe
    X_res, y_res = MLSMOTE(X_tfidf, y_sub, 100, indices)

    # Transform the test set
    X_test_tfidf = vectorize_test(X_test, count_vectorizer, tfidf_transformer)

    # Define the pipeline
    pipeline = Pipeline([
        ('clf', MultiOutputClassifier(estimator=RandomForestClassifier()))
    ])

    # Define the parameter grid
    param_grid = [
        {
            'clf__estimator__n_estimators': [100, 200],
            'clf__estimator__min_samples_split': [2, 5]
        },
        {
            'clf__estimator': [LogisticRegression(max_iter=1000)],
            'clf__estimator__C': [0.1, 1, 10],
            'clf__estimator__solver': ['liblinear', 'saga']
        }
    ]

    # Perform grid search
    grid_search = GridSearchCV(pipeline, param_grid, cv=2, scoring='precision_weighted', n_jobs=1, verbose=2)
    grid_search.fit(X_res, y_res)

    # Print the best parameters and best score
    print(f'Best parameters found: {grid_search.best_params_}')
    print(f'Best Precision score: {grid_search.best_score_}')

    # Save the best model
    joblib.dump(grid_search.best_estimator_, 'best_mode__type_of_support.pkl')
    joblib.dump(count_vectorizer, 'count_vectorizer.pkl')
    joblib.dump(tfidf_transformer, 'tfidf_transformer.pkl')

    # Predict on test data
    Y_pred = grid_search.best_estimator_.predict(X_test_tfidf)

    # Initialize lists to store the precision, recall, and f1-score for each label
    precision_list = []
    recall_list = []
    f1_list = []

    # Calculate precision, recall, and f1-score for each label
    for i, column in enumerate(Y.columns):
        precision = precision_score(y_test[column], Y_pred[:, i], average='weighted', zero_division=0)
        recall = recall_score(y_test[column], Y_pred[:, i], average='weighted', zero_division=0)
        f1 = f1_score(y_test[column], Y_pred[:, i], average='weighted', zero_division=0)

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)

    # Compute macro averages
    precision_macro = np.mean(precision_list)
    recall_macro = np.mean(recall_list)
    f1_macro = np.mean(f1_list)

    # Overall metrics
    overall_accuracy = (Y_pred == y_test).mean().mean()

    print(f'Overall Accuracy: {overall_accuracy:.4f}')
    print(f'Macro Average Precision: {precision_macro:.4f}')
    print(f'Macro Average Recall: {recall_macro:.4f}')
    print(f'Macro Average F1 Score: {f1_macro:.4f}')

if __name__ == "__main__":
    main()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sinde\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sinde\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
c:\Users\sinde\anaconda3\lib\site-packages\sklearn\neighbors\_base.py:584: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")


Fitting 2 folds for each of 10 candidates, totalling 20 fits


c:\Users\sinde\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[CV] END clf__estimator__min_samples_split=2, clf__estimator__n_estimators=100; total time= 1.7min
[CV] END clf__estimator__min_samples_split=2, clf__estimator__n_estimators=100; total time= 1.8min


c:\Users\sinde\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

[CV] END clf__estimator=LogisticRegression(max_iter=1000), clf__estimator__C=1, clf__estimator__solver=saga; total time=  15.0s


c:\Users\sinde\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[CV] END clf__estimator=LogisticRegression(max_iter=1000), clf__estimator__C=10, clf__estimator__solver=liblinear; total time=   1.4s
[CV] END clf__estimator=LogisticRegression(max_iter=1000), clf__estimator__C=10, clf__estimator__solver=liblinear; total time=   0.9s


c:\Users\sinde\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[CV] END clf__estimator=LogisticRegression(max_iter=1000), clf__estimator__C=10, clf__estimator__solver=saga; total time=  53.8s
[CV] END clf__estimator=LogisticRegression(max_iter=1000), clf__estimator__C=10, clf__estimator__solver=saga; total time=  28.6s
Best parameters found: {'clf__estimator__min_samples_split': 2, 'clf__estimator__n_estimators': 100}
Best Precision score: 0.6875401303112505
Overall Accuracy: 0.9149
Macro Average Precision: 0.9065
Macro Average Recall: 0.9149
Macro Average F1 Score: 0.8932
